In [1]:
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("../")


from plottingTools import utils

# Graph Greenifier

Predicting the performance and sustainability of a data center using simulation

This demo shows Greenifier function in the Graph Massivizer project

Greenifier can also be used as a separate tool

# 1. Topology

<img src="datacenter_image.jpg" alt="drawing" width="400"/>

Topologies define the available hardware of a datacenter

Defined using JSON format

[Example Topology](topologies/greenifier_topology.json)

# 2. Workloads

Workloads define what tasks need to be simulated and when

The workload used by Greenifier is similar to the output generated by Graph Optimizer

[Example workload](workloadTraces/greenifier/workload.json)

# 3. Carbon Intensity

To calculate the carbon emissions, we need information about the available energy mix

Greenifier gathers this information from the ENTSO-E project

In [2]:
df_carbon = pd.read_parquet("carbonTraces/carbon_2022.parquet")
df_carbon.head()

,timestamp,carbon_intensity
0,2021-12-31 23:00:00,168.138693
1,2021-12-31 23:15:00,167.050014
2,2021-12-31 23:30:00,164.552936
3,2021-12-31 23:45:00,167.493769
4,2022-01-01 00:00:00,164.517793


# 4. Scenario

Scenarios define what the Greenifier should simulate, and how

Scenarios are defined using a JSON format

[Example scenario](scenarios/greenifier_scenario.json)

# 5. Running Greenifier

Graph Greenifier can be run directly, using a terminal command

In [14]:
subprocess.run(["../bin/Greenifier", "--experiment-path", "experiments/experiment.json"])



 Running scenario: 0 
 Starting seed: 0 
GreenifierWorkloadSpec(tasks=[TaskSpec(name=pr, id=0, cpuCount=1, cpuUsage=10.000000 GHz, dependencies=[], memCapacity=976.562500 GiB, energyConsumption={H01=16692.242000 KJoule}, runTimes={H01=320}, submissionTime=2025-06-20), TaskSpec(name=find_max, id=1, cpuCount=1, cpuUsage=10.000000 GHz, dependencies=[0], memCapacity=976.562500 GiB, energyConsumption={H01=126.101000 KJoule}, runTimes={H01=2}, submissionTime=2025-06-20), TaskSpec(name=bfs, id=2, cpuCount=1, cpuUsage=10.000000 GHz, dependencies=[0], memCapacity=976.562500 GiB, energyConsumption={H01=1236.783000 KJoule}, runTimes={H01=29}, submissionTime=2025-06-20), TaskSpec(name=find_path, id=3, cpuCount=1, cpuUsage=10.000000 GHz, dependencies=[1, 2], memCapacity=976.562500 GiB, energyConsumption={H01=373.000000 Joule}, runTimes={H01=0}, submissionTime=2025-06-20), TaskSpec(name=pr, id=4, cpuCount=1, cpuUsage=10.000000 GHz, dependencies=[], memCapacity=976.562500 GiB, energyConsumption={H0

Simulating... 100% [=================================] 1/1 (0:00:00 / 0:00:00) 


CompletedProcess(args=['../bin/Greenifier', '--experiment-path', 'experiments/experiment.json'], returncode=0)

## 6. Output

In [15]:
pathToOutput = "output/greenifier"

df_host = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/host.parquet")
df_task = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/task.parquet")
df_service = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/service.parquet")

## 7. Aggregated results

To properly compare the different experiments, we aggregate them into more meaningful values.

### Performance

In [18]:
runtime = utils.getTotalRuntime(df_service) 
utilization = utils.getMeanUtilization(df_host)

print(f"The total runtime of the workload was {runtime}")
print(f"On average, the utilization of each host is {utilization * 100:.2f}%")

The total runtime of the workload was 0 days 00:00:02
On average, the utilization of each host is 0.00%


### Sustainability

In [21]:
energy_usage = utils.getTotalEnergyUsage(df_host, "kWh")
carbon_emissions = (df_host["carbon_emission"].sum() / 1000).round(2)

print(f"The data center used {energy_usage:.2f} kWh while running the workload")
print(f"The data center emitted {carbon_emissions:.2f} kg of carbon during the workload")

KeyError: 'carbon_emission'

## 8. Plotting results

Plotting data can provide further insights

In [20]:
utils.plotHosts(df_host, "carbon_emission", "sum", "carbon emissions")

KeyError: "Columns not found: 'carbon_emission'"

# 9. Graph Massivizer export

Greenifier provides the Graph-Choreographer with performance metrics.

These metrics are exported as a JSON file

In [19]:
output = utils.get_output(df_host, df_service, df_server, \
    save=True, exportName="output/greenifier_output.json")

[Exported File](output/greenifier_output.json)